# 09｜CNN 与 ViT：两种图像理解方式的结构对比

前面我们已经分别学习过 CNN 和 Tiny ViT。现在不再孤立地背结构，而是把它们放在同一张 `32 x 32` RGB 图片上，观察两种模型怎样逐步提取信息。

本课的核心结论不是谁一定更好，而是：

- **CNN 先在局部邻域提取特征，再逐层扩大感受野；**
- **ViT 先把图片变成 patch token，随后用 Self-Attention 直接建立全局关系。**

## 1. 为什么现在做对比

我们已经知道 CNN 的卷积、池化和特征图，也已经实现 ViT 的 Patch Embedding、CLS token、位置编码和 Encoder。现在正好可以回答：

1. CNN 的特征图与 ViT 的 token 序列是什么关系？
2. 两者怎样利用图片的空间结构？
3. 一处局部像素变化会怎样传播？
4. 为什么 ViT 往往比 CNN 更依赖数据量和训练策略？

> 注意：已有 CNN 使用 MNIST，TinyViT 使用 CIFAR-10。两个数据集难度、通道数和图片尺寸都不同，因此不能直接用两次训练准确率判断 CNN 与 ViT 谁更强。

In [1]:
import torch
from torch import nn

from vit import TinyViT

torch.manual_seed(42)
print("torch 版本：", torch.__version__)

torch 版本： 2.10.0


## 2. 先看两条完整数据流

### CNN

```text
B x 3 x 32 x 32
→ 卷积提取局部特征
→ B x C x H x W 特征图
→ 池化降低分辨率
→ 更深层卷积组合高级特征
→ 全局池化 / Flatten
→ B x 10
```

### ViT

```text
B x 3 x 32 x 32
→ 切成 8 x 8 = 64 个 patches
→ B x 64 x 256 patch tokens
→ 加 CLS 和位置编码
→ B x 65 x 256
→ 6 个 Transformer Encoder Blocks
→ 取 CLS token
→ B x 10
```

特征图中的一个空间位置和 patch 序列中的一个 token，都代表图片的一个区域。区别是它们后续交换信息的方式不同。

## 3. 建立一个同输入的小型 CNN

这里只建立结构参照物，不进行训练。它与 TinyViT 都接收 `B x 3 x 32 x 32`，都输出 `B x 10`。这样可以公平比较 shape，但**不能根据参数量或随机输出判断性能**。

In [2]:
class SmallCIFAR10CNN(nn.Module):
    """用于结构对照的小型 CNN：输入 Bx3x32x32，返回 Bx10 logits。"""

    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.features = nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.features(x)
        return self.classifier(x)

cnn = SmallCIFAR10CNN().eval()
vit = TinyViT().eval()
images = torch.randn(2, 3, 32, 32)

with torch.inference_mode():
    cnn_logits = cnn(images)
    vit_logits = vit(images)

print("CNN 输出：", tuple(cnn_logits.shape))
print("ViT 输出：", tuple(vit_logits.shape))
assert cnn_logits.shape == vit_logits.shape == (2, 10)

CNN 输出： (2, 10)
ViT 输出： (2, 10)


## 4. 同样的输入，不同的中间表示

使用 forward hook 记录中间模块的输出。hook 只观察输出，不会修改模型。

- CNN 保持 `B x C x H x W`，通道逐渐增加，高宽逐渐减小；
- ViT 使用 `B x N x D`，token 数量和维度在 Encoder 中保持不变。

In [3]:
cnn_shapes, vit_shapes = {}, {}

def save_shape(container, name):
    """返回 hook；参数 container 保存结果，name 是显示名称。"""
    def hook(module, inputs, output):
        container[name] = tuple(output.shape)
    return hook

hooks = [
    cnn.stem.register_forward_hook(save_shape(cnn_shapes, "第一层卷积")),
    cnn.features.register_forward_hook(save_shape(cnn_shapes, "卷积特征提取结束")),
    cnn.classifier.register_forward_hook(save_shape(cnn_shapes, "分类输出")),
    vit.input_embedding.patch_embedding.register_forward_hook(
        save_shape(vit_shapes, "Patch Embedding")
    ),
    vit.input_embedding.register_forward_hook(save_shape(vit_shapes, "加入 CLS 和位置编码")),
    vit.encoder.blocks[0].register_forward_hook(save_shape(vit_shapes, "第 1 个 Encoder Block")),
    vit.encoder.register_forward_hook(save_shape(vit_shapes, "6 个 Blocks 结束")),
    vit.classification_head.register_forward_hook(save_shape(vit_shapes, "分类输出")),
]

with torch.inference_mode():
    _ = cnn(images)
    _ = vit(images)
for hook in hooks:
    hook.remove()

print("CNN 中间 shape")
for name, shape in cnn_shapes.items():
    print(f"  {name:20s} -> {shape}")

print("\nViT 中间 shape")
for name, shape in vit_shapes.items():
    print(f"  {name:20s} -> {shape}")

assert cnn_shapes["卷积特征提取结束"] == (2, 64, 8, 8)
assert vit_shapes["Patch Embedding"] == (2, 64, 256)
assert vit_shapes["第 1 个 Encoder Block"] == (2, 65, 256)

CNN 中间 shape
  第一层卷积                -> (2, 32, 32, 32)
  卷积特征提取结束             -> (2, 64, 8, 8)
  分类输出                 -> (2, 10)

ViT 中间 shape
  Patch Embedding      -> (2, 64, 256)
  加入 CLS 和位置编码         -> (2, 65, 256)
  第 1 个 Encoder Block  -> (2, 65, 256)
  6 个 Blocks 结束        -> (2, 65, 256)
  分类输出                 -> (2, 10)


## 5. 核心实验：只修改左上角一个 patch

准备两张几乎相同的图片：第一张全为 0，第二张只把左上角 `4 x 4` 区域改成 1。然后比较三个位置：

1. CNN 第一层卷积：只有扰动附近的空间位置变化；
2. ViT Patch Embedding：因为修改范围正好对齐一个 patch，所以只有一个 patch token 变化；
3. ViT 第一个 Encoder Block：Self-Attention 允许每个 token 查看全部 token，因此变化可以立即传播到 CLS 和其他 patch tokens。

这就是局部感受野和全局注意力最直观的区别。

In [4]:
image_a = torch.zeros(1, 3, 32, 32)
image_b = image_a.clone()
image_b[:, :, :4, :4] = 1.0

with torch.inference_mode():
    cnn_a, cnn_b = cnn.stem(image_a), cnn.stem(image_b)
    patch_a = vit.input_embedding.patch_embedding(image_a)
    patch_b = vit.input_embedding.patch_embedding(image_b)
    input_a, input_b = vit.input_embedding(image_a), vit.input_embedding(image_b)
    block_a = vit.encoder.blocks[0](input_a)
    block_b = vit.encoder.blocks[0](input_b)

cnn_changed = ((cnn_a - cnn_b).abs().sum(dim=1)[0] > 1e-7)
patch_changed = ((patch_a - patch_b).abs().sum(dim=-1)[0] > 1e-7)
block_changed = ((block_a - block_b).abs().sum(dim=-1)[0] > 1e-7)

print(f"CNN 第一层发生变化的位置：{cnn_changed.sum().item()} / 1024")
print(f"Patch Embedding 后变化的 token：{patch_changed.sum().item()} / 64")
print(f"第一个 Encoder Block 后变化的 token：{block_changed.sum().item()} / 65")
print(f"其中 CLS token 是否变化：{bool(block_changed[0])}")

assert 1 < cnn_changed.sum().item() < 32 * 32
assert patch_changed.sum().item() == 1
assert block_changed.sum().item() > 1
assert bool(block_changed[0])

CNN 第一层发生变化的位置：25 / 1024
Patch Embedding 后变化的 token：1 / 64
第一个 Encoder Block 后变化的 token：65 / 65
其中 CLS token 是否变化：True


## 6. 归纳偏置：模型天生相信什么

归纳偏置可以理解为模型在看数据以前就具备的结构假设。

### CNN 的假设更强

- **局部性**：相邻像素通常比远处像素关系更密切；
- **平移等变性**：同一个卷积核会在整张图片上共享；
- **层次性**：边缘逐渐组合成纹理、部件和物体。

这些假设很符合自然图片，因此 CNN 在数据较少时通常更容易学起来。

### ViT 的限制更少

Self-Attention 不强制一个 token 只看邻居，而是通过数据学习哪些 token 应该联系。位置关系也不是由卷积网格天然提供，而需要位置编码补充。

限制少意味着表达更灵活，也意味着模型需要从数据中学习更多规律。这就是 ViT 往往更依赖数据增强、正则化、预训练和充分训练的原因。

## 7. 计算量怎样随图片变大

设 token 数量为 $N$，token 维度为 $D$。标准 Self-Attention 的注意力矩阵为 $N \times N$，核心计算量近似为：

$$O(N^2D)$$

图片高宽扩大时，patch 数量会增加，注意力计算按 token 数量的平方增长。当前 CIFAR-10 只有 64 个 patch，问题不大；高分辨率图片会更明显。

卷积只在 $K \times K$ 的局部窗口计算，单层计算量近似为：

$$O(HW K^2 C_{in}C_{out})$$

CNN 的局部计算更稳定，ViT 的全局建模更直接。后续 Swin Transformer 的 Window Attention，就是为了兼顾这两点。

## 8. 参数量只反映模型规模，不等于模型好坏

下面统计当前两个示例的可学习参数。小型 CNN 是为本课临时设计的，TinyViT 则是已经训练过的正式模型，因此这个数字只能说明规模差异，不能作为性能对比。

In [5]:
def count_parameters(model):
    """返回模型中 requires_grad=True 的参数总数。"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

cnn_parameters = count_parameters(cnn)
vit_parameters = count_parameters(vit)

print(f"示例 CNN 参数量：{cnn_parameters:,}")
print(f"当前 TinyViT 参数量：{vit_parameters:,}")
print(f"TinyViT 约为示例 CNN 的 {vit_parameters / cnn_parameters:.1f} 倍")

assert vit_parameters == 4_771_082

示例 CNN 参数量：20,042
当前 TinyViT 参数量：4,771,082
TinyViT 约为示例 CNN 的 238.1 倍


## 9. 最终对照表

| 对比维度 | CNN | ViT |
|---|---|---|
| 基本单位 | 像素网格上的局部窗口 | patch token |
| 中间表示 | `B x C x H x W` 特征图 | `B x N x D` token 序列 |
| 信息交换 | 卷积核连接局部邻域 | Attention 连接全部 token |
| 全局信息 | 逐层扩大感受野 | 一个 Attention 层即可全局交互 |
| 空间位置 | 卷积网格天然保留局部结构 | 依靠 token 顺序和位置编码 |
| 权重共享 | 同一卷积核扫描各位置 | 同一 QKV/MLP 处理各 token |
| 典型归纳偏置 | 局部性、平移等变性较强 | 结构限制较少、学习更灵活 |
| 数据需求 | 小数据上通常更容易训练 | 通常更依赖预训练和训练策略 |
| 高分辨率代价 | 局部卷积近似线性增长 | 全局 Attention 随 token 数平方增长 |

二者并不是完全对立：我们的 Patch Embedding 本身就使用了 `Conv2d`；现代视觉模型也经常混合卷积和 Attention。

## 10. 本课总结

1. CNN 把图片加工成多通道特征图，ViT 把图片加工成 token 序列；
2. CNN 的卷积先处理局部区域，感受野随网络加深逐渐扩大；
3. ViT 的 Patch Embedding 同样先局部切分，但 Self-Attention 会让 token 立即进行全局交互；
4. CNN 具有更强的图像归纳偏置，ViT 更灵活但通常需要更多数据和训练技巧；
5. 参数量、数据集和训练方案不一致时，不能只看准确率就判断架构优劣。

下一课将进入 Attention Map 可视化：直接观察训练好的 TinyViT 中，CLS token 对 64 个 patch 分配了怎样的注意力。

## 11. 自测问题

1. CNN 的 `B x C x H x W` 与 ViT 的 `B x N x D` 分别表示什么？
2. 为什么只修改一个 patch 后，Patch Embedding 只改变一个 token？
3. 为什么经过第一个 Attention Block 后，CLS token 也会改变？
4. CNN 的感受野为什么需要逐层扩大？
5. ViT 为什么需要位置编码，而 CNN 没有显式位置编码？
6. 为什么不能用 MNIST-CNN 与 CIFAR10-ViT 的准确率直接比较两种架构？
7. 标准 Self-Attention 在高分辨率图片上面临什么问题？